# Week 3 — What's on the desk? Context management for chatbots and agents

*Course project: a personal assistant for **Alex, 44**: senior backend developer, full-time job, two kids,
retraining as an **AI engineer** while job-hunting.*

This notebook follows the talk **"What's on the desk?"** section by section, with the talk's code
(the `context_toolkit` examples 01–09). Every section has three steps:

1. **The code from the slide**: read it, it's short.
2. **The talk's demo**: runs offline in a second, no API key needed.
3. **🧪 With Alex**: the same technique on Alex's real assistant, with a real model.

> **An LLM only sees what is on its desk.** It re-reads the whole desk every turn. The desk has a fixed size.
> Off the desk means gone. **Context management** is deciding what goes on the desk, what comes off, and in what
> form, *before* things drop off on their own.

| On the desk | In the system | Section |
|---|---|---|
| The desk | the context window: everything sent in one request | 1 |
| The brief pinned in the corner | system prompt and task | 2 |
| Papers you are working on | recent messages | 2, 3 |
| A one-page summary | compaction | 3 |
| Binders piled on top | tool results | 4 |
| A sticky note: "binder is on shelf 3" | offloading, handles, clearing old tool results | 4 |
| The drawer | memory: notes kept outside the window | 5 |
| An intern sent to the archive | a sub-agent with its own desk | 6 (optional) |
| The library | retrieval: just-in-time reads | 7 (optional) |

**Four moves:** *write* · *select* · *compress* · *isolate*. **Cheapest first:** offload at the door → clear → compact → reset.

⏱ **Core: sections 1–5, 8, 9 (~70 min).** Sections 6–7 and the 🧪 marked *optional* are for after class.

## 0. Setup

You did the setup at home (`SETUP.md`); this checks it again. This week runs on **OpenAI**: the techniques work
directly on OpenAI's list of conversation items. (Claude has the same ideas built into its API: context editing
and compaction.)

In [ ]:
%run ../check_setup.py --no-api

In [ ]:
import copy, hashlib, json, math, os, re, shutil, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable

import agent_core, lifedb, life_tools
from agent_core import WRITE, run_agent
from providers import load_api_key, open_conversation

PROVIDER, MODEL = "openai", "gpt-6-luna"
load_api_key(PROVIDER)
from openai import OpenAI
client = OpenAI()

DB = lifedb.create_db()          # Alex's week (dated to this week) + a year of history
WEEK = lifedb.week()
SYSTEM = (
    "You are a personal assistant for Alex, a 44-year-old software developer with a full-time job and two kids, "
    f"who is retraining to become an AI engineer. Today is {WEEK.today}. "
    "Use the tools to look at Alex's calendar, tasks and notes; never guess. Reply briefly."
)

print(f"Ready: {MODEL}, Alex's week {WEEK.label(0)} – {WEEK.label(6)}")

### Shared helpers (the toolkit's `common.py`)
*A measuring tape and a model to call.* Messages use a simple, provider-neutral chat shape:
`{"role": "user" | "assistant", "content": str | list[block]}`, where a block is `text`, `tool_use` or `tool_result`.
`make_llm(real=False)` is a deterministic **fake summariser** (offline demos); `make_llm(real=True)` calls OpenAI.

In [ ]:
Message = dict
LLM = Callable[[str], str]  # prompt in -> text out


def count_tokens(obj) -> int:
    """Rough token estimate (~4 characters per token for English text).

    Good enough to drive thresholds. For exact numbers read `usage.input_tokens`
    from the last API response."""
    text = obj if isinstance(obj, str) else json.dumps(obj, ensure_ascii=False)
    return max(1, len(text) // 4)


def context_size(messages: list[Message], system: str = "") -> int:
    return count_tokens(system) + sum(count_tokens(m) for m in messages)


KEY_PATTERNS = re.compile(r"(decid|agreed|must|never|always|deadline|budget|todo|open question|"
                          r"prefer|error|fixed|found|result|\d)", re.IGNORECASE)


def fake_llm(prompt: str) -> str:
    """Deterministic 'summarizer' so demos run without an API key: keeps sentences that look
    important (decisions, numbers, constraints) and drops chit-chat."""
    body = prompt.split("<transcript>")[-1].split("</transcript>")[0]
    sentences = re.split(r"(?<=[.!?])\s+|\n", body)
    keep = [s.strip() for s in sentences if s.strip() and KEY_PATTERNS.search(s)]
    seen, out = set(), []
    for s in keep:
        if s not in seen:
            seen.add(s)
            out.append(s)
    return "\n".join(f"- {s[:160]}" for s in out[:15]) or "- (nothing important)"


def make_llm(real: bool = False, model: str | None = None) -> LLM:
    """real=False: the offline fake summariser. real=True: an OpenAI model via the Responses API."""
    if not real:
        return fake_llm
    model = model or os.getenv("OPENAI_SUMMARY_MODEL", "gpt-6-luna")

    def call(prompt: str) -> str:
        return client.responses.create(model=model, input=prompt, max_output_tokens=1500).output_text

    return call


def render(messages: list[Message], width: int = 90) -> str:
    """One line per message - handy for showing 'what is on the desk'."""
    lines = []
    for m in messages:
        c = m["content"]
        if isinstance(c, str):
            lines.append(f"[{m['role']}] {c[:width]}")
            continue
        for b in c:
            t = b.get("type")
            if t == "text":
                lines.append(f"[{m['role']}] {b['text'][:width]}")
            elif t == "tool_use":
                lines.append(f"[{m['role']} -> tool] {b['name']}({json.dumps(b['input'])[:60]})")
            elif t == "tool_result":
                body = b["content"] if isinstance(b["content"], str) else json.dumps(b["content"])
                lines.append(f"[tool_result] ({count_tokens(body)} tok) {body[:width - 20]}")
    return "\n".join(lines)


def to_transcript(messages: list[Message], max_tool_chars: int = 300) -> str:
    """Flatten messages to plain text for a summarizer prompt."""
    out = []
    for m in messages:
        c = m["content"]
        if isinstance(c, str):
            out.append(f"{m['role'].upper()}: {c}")
            continue
        for b in c:
            t = b.get("type")
            if t == "text":
                out.append(f"{m['role'].upper()}: {b['text']}")
            elif t == "tool_use":
                out.append(f"TOOL CALL: {b['name']}({json.dumps(b['input'])})")
            elif t == "tool_result":
                body = b["content"] if isinstance(b["content"], str) else json.dumps(b["content"])
                out.append(f"TOOL RESULT: {body[:max_tool_chars]}")
    return "\n".join(out)

## 1. Token budget — measure the desk before you manage it
*How big is the desk, and what is already on it?* (slides 5, 11–12)

What's on the desk: **system prompt · tool definitions · memory · retrieved documents · conversation history ·
tool results · room reserved for the answer**. You can't fix what you don't measure.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 1 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
@dataclass
class ContextBudget:
    window: int = 200_000          # model's context window (the desk size)
    reserve_output: int = 8_000    # always leave room for the answer
    zones: dict = field(default_factory=dict)

    def add(self, zone: str, content) -> None:
        self.zones[zone] = self.zones.get(zone, 0) + count_tokens(content)

    @property
    def used(self) -> int:
        return sum(self.zones.values())

    @property
    def free(self) -> int:
        return self.window - self.reserve_output - self.used

    def pressure(self) -> float:
        """0.0 = empty desk, 1.0 = desk full (excluding answer reserve)."""
        return self.used / (self.window - self.reserve_output)

    def action(self) -> str:
        p = self.pressure()
        if p < 0.50:
            return "OK - nothing to do"
        if p < 0.70:
            return "CLEAR - drop old tool results (cheap, no LLM call)"
        if p < 0.85:
            return "COMPACT - summarise older history"
        return "RESET - write notes to memory, start a fresh context"

    def report(self) -> str:
        rows = [f"{'zone':<14}{'tokens':>10}{'share':>8}"]
        for z, t in sorted(self.zones.items(), key=lambda kv: -kv[1]):
            rows.append(f"{z:<14}{t:>10,}{t / max(self.used, 1):>8.0%}")
        rows.append(f"{'USED':<14}{self.used:>10,}")
        rows.append(f"{'FREE':<14}{self.free:>10,}   (after {self.reserve_output:,} reserved for output)")
        rows.append(f"pressure = {self.pressure():.0%}  ->  {self.action()}")
        return "\n".join(rows)

**The talk's demo** (offline):

In [ ]:
b = ContextBudget(window=200_000)
b.add("system", "You are a helpful banking assistant. " * 60)
b.add("tools", [{"name": f"tool_{i}", "description": "x" * 800} for i in range(25)])
b.add("memory", "Customer prefers Romanian. Risk profile: moderate. " * 20)
b.add("history", "user: ... assistant: ... " * 4_000)
b.add("tool_results", "{'rows': [...]} " * 30_000)   # one big SQL dump
print(b.report())

### 🧪 With Alex: an 8-week review
Alex asks for a review of the last 8 weeks. The assistant uses a careless tool that returns **everything** as JSON:
a binder per week. Before every model call (`before_call`), we put Alex's desk into a `ContextBudget`.

Our desk is **tiny on purpose** (6,000 tokens) so you can watch it fill up. Real models have 400k+ tokens, but agents
fill them the same way, just later, and pay for every token on every step.
(`parallel_tool_calls: False`: one week per step, so the desk grows step by step.)

In [ ]:
WINDOW, RESERVE = 6_000, 1_000

REVIEW = (f"Review my last 8 weeks, one week at a time (each Monday to Sunday, the first one starting "
          f"{WEEK.day(-56)}). For each week: hours spent in AI course sessions, number of networking chats, "
          "number of gym sessions. Look up exactly one week per tool call, and before the next lookup write "
          "that week's three numbers in one short line (your notes). "
          "Then tell me in two sentences which week was the least balanced.")


OPAQUE = ("encrypted_content", "id", "status")   # bookkeeping fields: not text the model reads


def desk_budget(conv) -> ContextBudget:
    """Alex's desk, zone by zone: what the model will read on its next call."""
    b = ContextBudget(window=WINDOW, reserve_output=RESERVE)
    b.add("system", conv.system)
    b.add("tools", conv.tools)
    for item in conv.input:
        kind = item.get("type")
        zone = ("tool_results" if kind == "function_call_output"
                else "tool_calls" if kind in ("function_call", "reasoning") else "history")
        b.add(zone, {k: v for k, v in item.items() if k not in OPAQUE})
    return b


def ground_truth(weeks=8):
    for k in range(weeks, 0, -1):
        start, end = WEEK.day(-7 * k), WEEK.day(-7 * k + 6)
        rows = life_tools.db_rows("SELECT title FROM events WHERE date BETWEEN ? AND ?", (start, end))
        course = 1.5 * sum(r["title"].startswith("AI course") for r in rows)
        chats = sum(r["title"] == "Networking chat" for r in rows)
        gym = sum(r["title"] == "Gym" for r in rows)
        print(f"week of {start}: course {course:4.1f} h · networking {chats} · gym {gym}")

In [ ]:
def watch_desk(conv, turn):
    b = desk_budget(conv)
    print(f"  step {turn:2}: desk {b.used:6,} tok · pressure {b.pressure():4.0%} → {b.action()}")

r_plain = run_agent(REVIEW, life_tools.BINDER_TOOLS, system=SYSTEM, model=MODEL, max_turns=15, verbose=False,
                    extra={"parallel_tool_calls": False}, before_call=watch_desk)
print("\n" + r_plain.answer + "\n")
print(desk_budget(r_plain.conv).report())
print(f"(the API counted {r_plain.steps[-1]['input']:,} input tokens on the last call: our estimate is rough but close)")

In [ ]:
ground_truth()

**Discuss:** What fills Alex's desk at the end: the chat, or the binders? (In Anthropic's cookbook run, 96% of an
agent's context was file contents it had already read.) At which step did the ladder say CLEAR? COMPACT?
Did the desk pass 100%? On a real model that's an error, or worse, the oldest papers silently fall off.

## 2. Sliding window — keep the newest papers, pin the brief
*The simplest strategy* (slide 13)

Keep only the most recent messages that fit a token budget. Two details make it production-safe:
1. **Pin** what must never fall off (the original task, the brief).
2. **Cut at safe boundaries:** never keep a tool result whose tool call fell off (the API rejects it).

And **say what was dropped**: a silent gap confuses the model. Limit: decisions made mid-conversation vanish.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 2 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
def _starts_with_orphan_tool_result(msg) -> bool:
    c = msg["content"]
    return msg["role"] == "user" and isinstance(c, list) and any(b.get("type") == "tool_result" for b in c)


def _text(msg) -> str:
    c = msg["content"]
    return c if isinstance(c, str) else " ".join(b.get("text", "") for b in c)


def sliding_window(messages, budget_tokens: int, pin_first: int = 1):
    """Return pinned head + the newest messages that fit in budget_tokens."""
    pinned = messages[:pin_first]
    budget = budget_tokens - sum(count_tokens(m) for m in pinned)

    kept = []
    for msg in reversed(messages[pin_first:]):
        cost = count_tokens(msg)
        if cost > budget:
            break
        kept.insert(0, msg)
        budget -= cost

    # Pairing rule: drop leading tool_results whose tool_use fell off the desk.
    while kept and _starts_with_orphan_tool_result(kept[0]):
        kept.pop(0)
    # Roles must alternate: the first kept message after the pin should be an assistant turn.
    while kept and pinned and kept[0]["role"] == pinned[-1]["role"]:
        kept.pop(0)

    dropped = len(messages) - len(pinned) - len(kept)
    if dropped:
        # Tell the model something is missing - silent gaps cause confusion.
        note = {"role": "assistant", "content": f"[{dropped} earlier messages were removed to save space]"}
        if kept and kept[0]["role"] == "assistant":
            kept[0] = {"role": "assistant", "content": note["content"] + "\n" + _text(kept[0])}
        else:
            kept.insert(0, note)
    return pinned + kept

**The talk's demo** (offline):

In [ ]:
convo = [{"role": "user", "content": "BRIEF: Plan a data migration for a retail bank. Budget 400k EUR, go-live 1 March."}]
for i in range(1, 30):
    convo.append({"role": "assistant", "content": f"Step {i}: analysis of workstream {i}. " + "detail " * 40})
    convo.append({"role": "user", "content": f"Thanks, continue with point {i + 1}."})

print(f"Full history : {len(convo)} messages, ~{context_size(convo):,} tokens")
trimmed = sliding_window(convo, budget_tokens=2_000)
print(f"After window : {len(trimmed)} messages, ~{context_size(trimmed):,} tokens\n")
print(render(trimmed, width=80))

The 🧪 with Alex for the sliding window is in section 3, side by side with compaction.

## 3. Compaction — the one-page summary
*Before papers fall off the desk, write a one-page summary* (slide 14)

When the context crosses a trigger (e.g. 70% of the window), summarise the **older** part and replace it with the
summary. The newest turns stay word for word. The summary is **incremental**: the previous summary goes in, so
knowledge accumulates instead of blurring. **The prompt is the product:** you decide what must survive.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 3 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
SUMMARY_PROMPT = """You are compacting the working memory of an AI assistant.
Write a summary that lets the assistant continue the task WITHOUT the original
messages. Keep, in this order:
1. Goal of the task and the user's constraints (budgets, dates, preferences)
2. Decisions already made, and why
3. Facts and numbers discovered (exact values, IDs, file names)
4. What has been done / what is still open (next steps, open questions)
Drop: greetings, repetition, raw tool output already digested.
Be concise; use bullet points.

<previous_summary>
{previous}
</previous_summary>

<transcript>
{transcript}
</transcript>

Now write the summary. Do NOT answer or continue the conversation above, even if it ends with a question."""
# ^ course fix: without that last line, a real model sometimes answered the transcript's last question
#   instead of summarising it, and every earlier fact was lost (see the discussion below).

SUMMARY_TAG = "[CONVERSATION SUMMARY]"


def _is_summary(m) -> bool:
    return isinstance(m["content"], str) and m["content"].startswith(SUMMARY_TAG)


class Compactor:
    def __init__(self, llm, window: int, trigger: float = 0.70, keep_last: int = 6):
        self.llm, self.window, self.trigger, self.keep_last = llm, window, trigger, keep_last
        self.summary = ""
        self.compactions = 0

    def needs_compaction(self, messages) -> bool:
        return context_size(messages) > self.trigger * self.window

    def compact(self, messages):
        if not self.needs_compaction(messages) or len(messages) <= self.keep_last:
            return messages
        head, tail = messages[:-self.keep_last], messages[-self.keep_last:]
        # Do not split a tool_use / tool_result pair across the boundary.
        while tail and tail[0]["role"] == "user" and isinstance(tail[0]["content"], list):
            head.append(tail.pop(0))
        # Messages must alternate and start with the user: make the tail start with assistant.
        if tail and tail[0]["role"] == "user":
            head.append(tail.pop(0))

        head = [m for m in head if not _is_summary(m)]  # old summary is passed separately
        self.summary = self.llm(SUMMARY_PROMPT.format(
            previous=self.summary or "(none)", transcript=to_transcript(head)))
        self.compactions += 1
        summary_msg = {"role": "user", "content": f"{SUMMARY_TAG}\n{self.summary}"}
        return [summary_msg] + tail

**The talk's demo** (offline, with the fake summariser):

In [ ]:
c = Compactor(make_llm(real=False), window=3_500, trigger=0.70, keep_last=4)
script = [
    ("user", "We need to migrate 12 million customer records from the legacy core to the new platform."),
    ("assistant", "Understood. Let's start with scope. How many source systems?"),
    ("user", "Three source systems. Budget is 400k EUR and the deadline is 1 March 2027."),
    ("assistant", "Great. I suggest a phased approach: pilot, wave 1, wave 2."),
    ("user", "Agreed, we decided on 3 waves. Also: never move PII outside the EU."),
    ("assistant", "Noted. Data quality check found 4.2% duplicate customer IDs."),
    ("user", "How nice, the weather is lovely today by the way."),
    ("assistant", "Glad to hear it! Next, reconciliation rules."),
    ("user", "Reconciliation must match balances to the cent. Open question: who signs off?"),
    ("assistant", "I'll propose the CFO office. TODO: draft the cut-over runbook."),
]
messages = []
for i in range(6):
    for role, text in script:
        messages.append({"role": role, "content": text + " " + "(filler) " * 25})
        messages = c.compact(messages)
    print(f"round {i}: {len(messages):>3} msgs, ~{context_size(messages):>5,} tokens, compactions so far = {c.compactions}")
print("\n--- The running summary ---\n" + c.summary)

### 🧪 With Alex: does the assistant remember the week?
Alex chats with the assistant all week (10 messages, Monday to Thursday). Some things said on Monday matter on
Friday. We replay the week, then ask **5 probe questions** about facts said earlier.
Three strategies: keep everything · `sliding_window` · `Compactor` with a **real** model.

In [ ]:
CHAT_SYSTEM = (
    "You are Alex's personal assistant. Alex is a 44-year-old developer with a full-time job and two kids, "
    f"retraining as an AI engineer. This chat runs through the week that starts on {WEEK.today}; "
    "every message starts with its day and time. Keep answers short: at most 80 words."
)

ALEX_WEEK = [
    ("Mon 07:45", "Morning! Context for this week: my partner is travelling Wednesday and Thursday, so I'm on my own "
                  "with the kids. No calls between 18:00 and 20:00 on those days."),
    ("Mon 12:30", "Lunch break. In what order should I tackle my capstone tasks? The RAG pipeline scares me."),
    ("Mon 21:10", "Coffee chat with Priya went great. She'll refer me to Kestrel Labs if I send her my resume by Friday."),
    ("Tue 12:10", "Give me 3 good questions to ask the Northwind recruiter tonight."),
    ("Tue 21:15", "Recruiter screen done! Next step: a take-home assignment, due next Wednesday, about 5 hours of work."),
    ("Wed 07:10", "Dentist in an hour. Honestly, what should I prioritise today?"),
    ("Wed 22:00", "Mock interview feedback: my STAR stories are too long. I should aim for 2 minutes each."),
    ("Thu 06:40", "Skipping the gym this morning to work on the RAG pipeline. Bad idea?"),
    ("Thu 21:30", "The RAG pipeline works! Retrieval accuracy went from 62% to 81% after I added reranking."),
    ("Thu 22:15", "Draft 3 bullet points for tomorrow's demo intro, please."),
]

PROBES = [
    ("(Fri 08:00) Remind me: on which evenings was I unavailable for calls this week, and from when to when?",
     lambda a: "18" in a and "20" in a),
    ("(Fri 08:05) By when do I need to send my resume to Priya?", lambda a: "friday" in a or "today" in a),
    ("(Fri 08:10) When is the Northwind take-home due, and how long will it take?",
     lambda a: "wednesday" in a and ("5" in a or "five" in a)),
    ("(Fri 08:15) What was the feedback from my mock interview?", lambda a: "2 minutes" in a or "two minutes" in a),
    ("(Fri 08:20) What retrieval accuracy did my RAG pipeline reach?", lambda a: "81" in a),
]


class ChatSession:
    """A chatbot: the desk is the list of messages. A strategy may tidy it before every call."""

    def __init__(self, strategy=None, extra=None, native=False):
        self.messages, self.strategy, self.extra, self.native, self.log = [], strategy, extra or {}, native, []

    def say(self, text: str) -> str:
        self.messages.append({"role": "user", "content": text})
        if self.strategy:
            self.messages = self.strategy(self.messages)
        r = client.responses.create(model=MODEL, instructions=CHAT_SYSTEM, input=self.messages,
                                    max_output_tokens=2000, reasoning={"effort": "low"}, **self.extra)
        cached = getattr(r.usage.input_tokens_details, "cached_tokens", 0) or 0
        self.log.append({"input": r.usage.input_tokens, "cached": cached})
        if self.native:     # keep the API's own items (needed for server-side compaction, section 8)
            self.messages += [o.model_dump(exclude_none=True, warnings=False) for o in r.output]
        else:
            self.messages.append({"role": "assistant", "content": r.output_text})
        return r.output_text


def play_and_probe(session, label):
    for when, text in ALEX_WEEK:
        session.say(f"({when}) {text}")
    score, answers = 0, []
    for question, check in PROBES:
        answer = session.say(question)
        ok = check(answer.lower())
        score += ok
        answers.append(f"  {'✅' if ok else '❌'} {question}\n     → {answer.strip()[:150]}")
    peak = max(x["input"] for x in session.log)
    total = sum(x["input"] for x in session.log)
    print(f"── {label}: probes {score}/5 · biggest desk {peak:,} tok · {total:,} input tokens in total")
    print("\n".join(answers))
    return {"label": label, "score": score, "peak": peak, "total": total}

Now the experiment (about 2 minutes: 3 × 15 messages). The desk is small on purpose again.

In [ ]:
compactor = Compactor(make_llm(real=True), window=850, trigger=0.70, keep_last=4)
chat_results = [
    play_and_probe(ChatSession(), "1. keep everything"),
    play_and_probe(ChatSession(strategy=lambda m: sliding_window(m, budget_tokens=500)), "2. sliding window"),
    play_and_probe(ChatSession(strategy=compactor.compact), "3. compaction"),
]

In [ ]:
print(f"{'strategy':22} {'probes':>6} {'biggest desk':>13} {'total input':>12}")
for c in chat_results:
    print(f"{c['label']:22} {c['score']:>4}/5 {c['peak']:>11,} t {c['total']:>10,} t")
print(f"\n{compactor.compactions} compactions. The final summary:\n" + compactor.summary)

**Discuss:**
- Which facts did the sliding window lose? Which survived, and why? (Hint: `pin_first=1` pins Monday's first message.)
- Read the summary. Is every probe fact in it? If not, change `SUMMARY_PROMPT` and rerun.
- **The prompt is the product.** Look at the last line of `SUMMARY_PROMPT`. Without it, in our test runs the model
  sometimes *answered* the last question in the transcript instead of summarising it, and every earlier fact vanished
  from the summary. Delete that line and rerun: how many probes survive?
- With a tiny desk the compactor runs ~10 times: a summary of a summary of a summary. Which detail blurred first?
- **Poisoning** (slide 7) is the scariest failure: a wrong fact that survives into the summary becomes "truth".

**(Optional) Poisoning: one wrong number in the summary.**

In [ ]:
poisoned_summary = compactor.summary.replace("81%", "91%")
print("poisoned:", poisoned_summary != compactor.summary)
poisoned = ChatSession()
poisoned.messages = [{"role": "user", "content": f"{SUMMARY_TAG}\n{poisoned_summary}"},
                     {"role": "assistant", "content": "Got it."}]
print(poisoned.say("(Fri 09:00) I'm writing my demo slides. What retrieval accuracy should I put on the results slide?"))

## 4. Tool output — a sticky note, not the binder
*For agents, tool output is usually 80–95% of the context* (slide 15)

Two techniques that cost **no model call**:
- **A. Offload at the door:** a huge tool output is saved to disk; the model gets a preview + a **handle**,
  and tools to look inside (`grep`, `read_slice`).
- **B. Clear old results:** after the agent has used a result, replace its body with a placeholder. The **call record
  stays**, so the model knows the call happened and can repeat it. **Never clear memory.**

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 4 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
STORE = Path("./.tool_outputs")
PLACEHOLDER = "[result cleared to save context - call the tool again if needed]"


def _stored(handle: str) -> Path:                     # course fix: a handle can't point outside STORE
    path = (STORE / handle).resolve()
    if not path.is_relative_to(STORE.resolve()):
        raise ValueError("handle escapes the tool output store")
    return path


# ---------- A. Offload large outputs at ingestion time ----------------------
def offload_if_large(tool_name: str, output: str, max_tokens: int = 500) -> str:
    """Return output unchanged if small; else save it and return a preview."""
    if count_tokens(output) <= max_tokens:
        return output
    STORE.mkdir(exist_ok=True)
    handle = f"{tool_name}-{hashlib.sha1(output.encode()).hexdigest()[:8]}.txt"
    (STORE / handle).write_text(output)
    lines = output.splitlines()
    head, tail = "\n".join(lines[:5]), "\n".join(lines[-3:])
    return (f"[Large output: {count_tokens(output):,} tokens, {len(lines):,} lines. "
            f"Saved as '{handle}'. Use read_slice(handle, start, end) or "
            f"grep(handle, pattern) to look inside.]\n"
            f"--- first lines ---\n{head}\n...\n--- last lines ---\n{tail}")


def grep(handle: str, pattern: str, limit: int = 20) -> str:
    """A tool the agent can call to search an offloaded output."""
    hits = [l for l in _stored(handle).read_text().splitlines() if pattern.lower() in l.lower()]
    return "\n".join(hits[:limit]) + (f"\n... {len(hits) - limit} more" if len(hits) > limit else "")


def read_slice(handle: str, start: int, end: int) -> str:       # course addition: the note promises it
    """A tool the agent can call to read lines start..end (1-based) of an offloaded output."""
    lines = _stored(handle).read_text().splitlines()
    return "\n".join(f"{i}: {l}" for i, l in enumerate(lines[start - 1:end], start))


# ---------- B. Clear old tool results ---------------------------------------
def clear_old_tool_results(messages, keep: int = 3, exclude_tools=("memory",)):
    """Replace all but the `keep` most recent tool results with a placeholder."""
    names = {}  # tool_use_id -> tool name
    for m in messages:
        if m["role"] == "assistant" and isinstance(m["content"], list):
            for b in m["content"]:
                if b.get("type") == "tool_use":
                    names[b["id"]] = b["name"]
    positions = [(mi, bi) for mi, m in enumerate(messages)
                 if m["role"] == "user" and isinstance(m["content"], list)
                 for bi, b in enumerate(m["content"])
                 if b.get("type") == "tool_result" and names.get(b["tool_use_id"]) not in exclude_tools]
    freed = 0
    for mi, bi in positions[:-keep] if keep else positions:
        block = messages[mi]["content"][bi]
        if block["content"] != PLACEHOLDER:
            freed += count_tokens(block["content"])
            block["content"] = PLACEHOLDER
    return freed

**The talk's demo** (offline): 5 SQL dumps.

In [ ]:
def fake_sql(table: str) -> str:
    rows = [json.dumps({"id": i, "table": table, "balance": i * 13.37, "status": "OK" if i % 97 else "MISMATCH"})
            for i in range(2_000)]
    return "\n".join(rows)

msgs = [{"role": "user", "content": "Reconcile the 5 account tables and report mismatches."}]
for i, table in enumerate(["current", "savings", "loans", "cards", "deposits"]):
    msgs.append({"role": "assistant", "content": [{"type": "tool_use", "id": f"t{i}", "name": "run_sql", "input": {"table": table}}]})
    msgs.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": f"t{i}", "content": fake_sql(table)}]})

print(f"Raw agent context          : ~{context_size(msgs):>8,} tokens")
cleared = copy.deepcopy(msgs)
freed = clear_old_tool_results(cleared, keep=1)
print(f"After clearing (keep=1)    : ~{context_size(cleared):>8,} tokens  (freed {freed:,})")
offloaded_msgs = copy.deepcopy(msgs)
for m in offloaded_msgs:
    if isinstance(m["content"], list):
        for b in m["content"]:
            if b.get("type") == "tool_result":
                b["content"] = offload_if_large("run_sql", b["content"])
print(f"With offload at ingestion  : ~{context_size(offloaded_msgs):>8,} tokens\n")
handle = next(p.name for p in STORE.iterdir() if "run_sql" in p.name)
print(f"grep('{handle}', 'MISMATCH'):\n{grep(handle, 'MISMATCH', limit=3)}")

### On OpenAI's items (from the talk's 08)
Alex's assistant runs on the OpenAI Responses API, where a tool call is a `function_call` item and its result a
`function_call_output` item. Same idea, different envelope: the talk's 08 has this version.

In [ ]:
CLEARED = "[output cleared to save context - call the tool again if needed]"


def clear_old_function_outputs(items, keep=4, exclude=("memory",)):
    """Example 04's clearing, for Responses API items.

    A tool call is a {"type": "function_call", "call_id", "name", ...} item;
    its result is a {"type": "function_call_output", "call_id", "output"} item.
    Keep the call record, blank all but the newest `keep` outputs."""
    names = {i["call_id"]: i["name"] for i in items if i.get("type") == "function_call"}
    outputs = [i for i in items
               if i.get("type") == "function_call_output" and names.get(i["call_id"]) not in exclude]
    for item in outputs[:-keep] if keep else outputs:
        item["output"] = CLEARED


def run_review(tools, manage=None, label="", max_turns=30):
    log = []
    def before_call(conv, turn):
        if manage:
            manage(conv.input)
        log.append(desk_budget(conv).used)
    r = run_agent(REVIEW, tools, system=SYSTEM, model=MODEL, max_turns=max_turns, verbose=False,
                  extra={"parallel_tool_calls": False}, before_call=before_call)
    return {"label": label, "log": log, "result": r}


def bars(run, scale=150):
    limit = WINDOW - RESERVE
    print(f"\n{run['label']}  (peak {max(run['log']):,} tokens, {run['result'].turns} steps)")
    for i, n in enumerate(run["log"], 1):
        print(f"  step {i:2} {n:6,} {'█' * (n // scale)}{'  ! over the desk' if n > limit else ''}")

### 🧪 With Alex: the review, with and without clearing (about 1 minute each)

In [ ]:
runs = [
    run_review(life_tools.BINDER_TOOLS, label="A. no management"),
    run_review(life_tools.BINDER_TOOLS, manage=lambda items: clear_old_function_outputs(items, keep=2),
               label="B. clear old results (keep 2)"),
]
for run in runs:
    bars(run)

In [ ]:
for run in runs:
    r = run["result"]
    print(f"── {run['label']}: {r.turns} steps, {r.tool_calls.count('list_events_raw')} lookups, "
          f"{r.input_tokens:,} input tokens in total\n{r.answer}\n")
ground_truth()

**(Optional) C. Offload at the door.** Every binder goes to disk; the model gets the sticky note and must use
`grep` / `read_slice`. Watch the number of steps: offloading trades desk space for **extra calls**. It pays off when
outputs are huge (logs, files, web pages) and the model needs only a few lines.

In [ ]:
STICKY_TOOLS = {
    "grep": {"fn": grep, "effect": agent_core.READ, "schema": {
        "name": "grep", "description": "Search a saved tool output (by its handle) for lines containing a text.",
        "input_schema": {"type": "object", "properties": {"handle": {"type": "string"}, "pattern": {"type": "string"}},
                         "required": ["handle", "pattern"]}}},
    "read_slice": {"fn": read_slice, "effect": agent_core.READ, "schema": {
        "name": "read_slice", "description": "Read lines start..end (1-based) of a saved tool output.",
        "input_schema": {"type": "object", "properties": {"handle": {"type": "string"}, "start": {"type": "integer"},
                                                          "end": {"type": "integer"}},
                         "required": ["handle", "start", "end"]}}},
}


def offloaded(tools: dict) -> dict:
    """Wrap every tool so big outputs are offloaded at the door, and add the sticky-note tools."""
    wrapped = {name: {**e, "fn": (lambda _fn=e["fn"], _name=name, **kw: offload_if_large(_name, _fn(**kw), max_tokens=150))}
               for name, e in tools.items()}
    return {**wrapped, **STICKY_TOOLS}

In [ ]:
run_c = run_review(offloaded(life_tools.BINDER_TOOLS), manage=lambda items: clear_old_function_outputs(items, keep=2),
                   label="C. offload at the door + clear", max_turns=40)
bars(run_c)
print(run_c["result"].tool_calls)
print(run_c["result"].answer)

**Discuss:**
- Which run kept the desk smallest? Which got the numbers right? Cheaper is only better if the answer survives.
- **Lookups:** 8 means the model trusted its notes; more means it fetched cleared weeks again (the placeholder says
  "call the tool again if needed"). Run B twice: the same code can give either. Clearing is safe *because* the tool
  can be called again, but each re-fetch costs a step.
- Clearing only works because the model **wrote its notes** first. Remove "write that week's three numbers..." from
  `REVIEW` and run B again. (Write it down, before it falls off.)

## 5. Memory — the drawer next to the desk
*The only thing that survives a reset or a new session* (slide 16)

Compaction and clearing work **inside** the window. Memory moves things **out** of it. The model decides what to
write, through a `memory` tool; we host the storage. Two flavours: progress notes for a long task (agents), and small
user facts, loaded only when relevant (chatbots).

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 5 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
MEMORY_TOOL_SCHEMA = {
    "name": "memory",
    "description": (
        "Your persistent notes. ALWAYS view /memories at the start of a task. "
        "Write down decisions, progress and facts you will need later - your "
        "context may be reset at any moment. Keep files short and organised."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "command": {"type": "string", "enum": ["view", "write", "append", "search", "delete"]},  # course fix: type
            "path": {"type": "string"},
            "text": {"type": "string"},
            "query": {"type": "string"},
        },
        "required": ["command"],
    },
}


class FileMemory:
    """Client-side backend. The model calls it through the tool above."""

    def __init__(self, root="./memories", max_file_tokens=2_000):
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_file_tokens = max_file_tokens

    def _path(self, p: str) -> Path:
        rel = p.removeprefix("/memories").lstrip("/") or "."
        full = (self.root / rel).resolve()
        if not full.is_relative_to(self.root):      # course fix: block ../../etc/passwd AND ../memories_evil
            raise ValueError("path escapes memory directory")
        return full

    def handle(self, cmd: dict) -> str:
        try:
            c = cmd["command"]
            if c == "view":
                p = self._path(cmd.get("path", "/memories"))
                if p.is_dir():   # course fix: show each note's content, not just its name (the model
                                 # often stopped at the list of names and never opened the note)
                    notes = [f"## {x.name}\n{x.read_text()}" for x in sorted(p.iterdir()) if x.is_file()]
                    return "\n\n".join(notes) or "(empty - nothing remembered yet)"
                return p.read_text()
            if c in ("write", "append"):
                p = self._path(cmd["path"])
                p.parent.mkdir(parents=True, exist_ok=True)
                new = (p.read_text() + "\n" if c == "append" and p.exists() else "") + cmd["text"]
                if count_tokens(new) > self.max_file_tokens:
                    return "Error: file too long - summarise it before writing more."
                p.write_text(new)
                return f"Saved {cmd['path']} ({count_tokens(new)} tokens)"
            if c == "search":
                q = cmd["query"].lower()
                hits = [f"{f.name}: {line.strip()}" for f in self.root.rglob("*.md")
                        for line in f.read_text().splitlines() if q in line.lower()]
                return "\n".join(hits[:20]) or "no matches"
            if c == "delete":
                self._path(cmd["path"]).unlink()
                return f"Deleted {cmd['path']}"
            return (f"Error: unknown command {c}. Commands: view (a folder, or a file to read it), "   # course fix:
                    "write, append, search, delete.")                                                   # say what to do
        except Exception as e:  # errors go back to the model, not up the stack
            return f"Error: {e}"


def load_relevant_memories(memory: FileMemory, user_message: str, budget_tokens=400) -> str:
    """For chatbots: inject only memories that match the new message."""
    words = {w.strip(".,?!").lower() for w in user_message.split() if len(w) > 3}
    scored = []
    for f in memory.root.rglob("*.md"):
        for line in f.read_text().splitlines():
            score = sum(w in line.lower() for w in words)
            if score:
                scored.append((score, line.strip()))
    out, used = [], 0
    for _, line in sorted(scored, reverse=True):
        if used + count_tokens(line) > budget_tokens:
            break
        out.append(line)
        used += count_tokens(line)
    return "\n".join(out)

**The talk's demo** (offline): two sessions, plus the path guard.

In [ ]:
shutil.rmtree("./memories_demo", ignore_errors=True)
demo = FileMemory("./memories_demo")
for call in [{"command": "view", "path": "/memories"},
             {"command": "write", "path": "/memories/progress.md", "text": "# Migration task\n- [x] Wave 1 mapped\n- [ ] Wave 2: loans, cards"},
             {"command": "write", "path": "/memories/decisions.md", "text": "- Decided: 3 migration waves\n- Budget: 400k EUR"}]:
    print(f"memory.{call['command']:<6} -> {demo.handle(call)}")
print("\n=== New session, empty context ===\n" + demo.handle({"command": "view", "path": "/memories"}))
q = "What was the budget we agreed for the migration?"
print(f"\nUser asks: {q}\nInjected memory:\n{load_relevant_memories(demo, q)}")
print("\nPath traversal attempt ->", demo.handle({"command": "view", "path": "/memories/../../etc/passwd"}))

### 🧪 With Alex: something that matters on Thursday
**Monday, conversation 1:** Alex mentions something. **Thursday, conversation 2:** a fresh desk. Same question,
with and without the drawer.

In [ ]:
MEMORY = FileMemory("./memories")
MEMORY_TOOL = {"memory": {"fn": lambda **cmd: MEMORY.handle(cmd), "effect": WRITE, "schema": MEMORY_TOOL_SCHEMA}}
QUESTION = "Northwind asked for a quick call on Thursday from 7:45 to 8:15, before work. Can I say yes?"

In [ ]:
shutil.rmtree("./memories", ignore_errors=True); MEMORY = FileMemory("./memories")        # an empty drawer
r1 = run_agent("Heads-up for this week: my partner is travelling Wednesday and Thursday, so I'm doing the "
               "school drop-off at 8:00 on both days. I can't take anything between 7:30 and 8:30 then. "
               "Please remember that.",
               {**life_tools.TOOLS, **MEMORY_TOOL}, system=SYSTEM, model=MODEL)
print("\n" + r1.answer)
print("\n📂 In the drawer:", MEMORY.handle({"command": "view", "path": "/memories"}))

In [ ]:
r_no_memory = run_agent(QUESTION, life_tools.TOOLS, system=SYSTEM, model=MODEL, verbose=False)
r_memory = run_agent(QUESTION, {**life_tools.TOOLS, **MEMORY_TOOL}, system=SYSTEM, model=MODEL, verbose=False)
print("── without memory:", r_no_memory.tool_calls, "\n" + r_no_memory.answer)
print("\n── with memory:   ", r_memory.tool_calls, "\n" + r_memory.answer)

**Discuss:**
- Did the assistant open the drawer first? (Viewing the folder shows every note's content: with only the file
  names, the model often stopped there, or tried a `read` command that doesn't exist.)
- Remove "Please remember that." and run conversation 1 a few times. Does it always write a note? When the model
  decides what to remember, sometimes it doesn't.
- **Summary vs memory:** a summary is in-session and lossy; memory is cross-session and exact for what was saved.
- **Privacy:** memory is personal data: it needs a purpose, a retention period and a way to forget (`delete`).
  It's also an attack surface: notice the path guard.

## 6. (Optional) Sub-agents — send an intern to the archive
*Isolate* (slide 17). A sub-agent gets a **fresh, empty** context, does the messy work there, and returns only a
short, structured result: **the contract**. The orchestrator's desk stays clean.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 6 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
RETURN_BUDGET = 300  # tokens a sub-agent may hand back


def read_big_source(name: str) -> str:
    """Stand-in for an expensive tool: 30k tokens of raw data."""
    return "\n".join(f"{name} row {i}: amount={i * 7 % 1000} flag={'LATE' if i % 211 == 0 else 'ok'}"
                     for i in range(4_000))


def run_subagent(task: str, source: str) -> dict:
    """Everything inside this function lives in the sub-agent's own context."""
    sub_context = [{"role": "user", "content": task}]
    raw = read_big_source(source)
    sub_context.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": "x", "content": raw}]})
    late = [l for l in raw.splitlines() if "LATE" in l]     # the sub-agent 'thinks' (here: plain Python)
    result = {"source": source, "rows_scanned": len(raw.splitlines()), "late_rows": len(late),
              "examples": late[:3], "confidence": "high"}
    assert count_tokens(json.dumps(result)) <= RETURN_BUDGET, "sub-agent answer too long - summarise harder"
    result["_subagent_context_tokens"] = context_size(sub_context)
    return result

**The talk's demo** (offline):

In [ ]:
sources = ["payments_2024", "payments_2025", "standing_orders", "direct_debits"]
task = "Find late payments in {src}. Return JSON: rows_scanned, late_rows, 3 examples."
inline = [{"role": "user", "content": "Audit late payments across all sources."}]
for s in sources:
    inline.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": s, "content": read_big_source(s)}]})
print(f"Inline (one agent)       : orchestrator context ~{context_size(inline):,} tokens")
with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(lambda s: run_subagent(task.format(src=s), s), sources))
orchestrator = [{"role": "user", "content": "Audit late payments across all sources."}]
for r in results:
    r.pop("_subagent_context_tokens")
    orchestrator.append({"role": "user", "content": json.dumps(r)})
print(f"With sub-agents          : orchestrator context ~{context_size(orchestrator):,} tokens")

### 🧪 With Alex: one intern (a real model) per week of the review

In [ ]:
WEEK_PAGE = {
    "type": "object",
    "properties": {"course_hours": {"type": "number"}, "networking_chats": {"type": "integer"},
                   "gym_sessions": {"type": "integer"}, "note": {"type": "string"}},
    "required": ["course_hours", "networking_chats", "gym_sessions", "note"],
    "additionalProperties": False,
}


def alex_intern(k: int) -> dict:
    start, end = WEEK.day(-7 * k), WEEK.day(-7 * k + 6)
    binder = life_tools.list_events_raw(start, end)                                # the messy work
    intern = open_conversation(PROVIDER, MODEL, f"Events of the week {start} to {end}:\n{binder}", effort="low",
                               system="Count: hours of 'AI course' sessions (1.5 h each), number of 'Networking chat' "
                                      "events, number of 'Gym' events. note: one short sentence about the week.")
    page = {"week": start, **intern.complete_json(WEEK_PAGE)}
    assert count_tokens(json.dumps(page)) <= RETURN_BUDGET, "the contract: one small page"
    return page

In [ ]:
with ThreadPoolExecutor(max_workers=4) as pool:
    pages = list(pool.map(alex_intern, range(8, 0, -1)))
for p in pages:
    print(p)
final = client.responses.create(model=MODEL, reasoning={"effort": "low"},
                                input="Which of these weeks was the least balanced between the AI course, networking "
                                      f"and the gym? Two sentences.\n{json.dumps(pages, indent=1)}")
print(f"\nThe orchestrator's desk: ~{count_tokens(pages):,} tokens of pages "
      f"(vs ~{max(runs[0]['log']) if 'runs' in globals() else '?'} for one agent reading every binder)\n")
print(final.output_text)

**Discuss:** More total tokens (each intern pays for its own desk), but a sharp main agent and parallel work.
Good fit: research fan-out, per-file analysis. Poor fit: tightly coupled work where interns need each other's context.

## 7. (Optional) Retrieval — bring three pages, not the library
*Select.* Two ways to get knowledge onto the desk: **up-front** (retrieve the top-k chunks before the model runs) or
**just-in-time** (give the model a search tool, like coding agents use grep instead of reading the whole repo).
A tiny BM25 in pure Python; in production you'd swap in embeddings or a vector database. Same idea.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 7 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
DOCS = {
    "gdpr_art17.md": "Right to erasure. The data subject has the right to obtain erasure of personal data without undue delay.",
    "gdpr_art33.md": "Breach notification. The controller notifies the supervisory authority within 72 hours of becoming aware of a personal data breach.",
    "gdpr_art35.md": "A data protection impact assessment (DPIA) is required where processing is likely to result in a high risk.",
    "aiact_art50.md": "Transparency: providers ensure people are informed that they are interacting with an AI system.",
    "migration_runbook.md": "Cut-over weekend: freeze legacy writes Friday 18:00, reconcile balances, go/no-go Sunday 12:00.",
    "hr_policy.md": "Employees accrue 21 days of annual leave. Meal vouchers are granted per worked day.",
} | {f"filler_{i}.md": "Unrelated corporate text about office plants and parking. " * 30 for i in range(40)}


def tokenize(t: str):
    return re.findall(r"[a-z0-9]+", t.lower())


class BM25:
    def __init__(self, docs: dict, k1=1.5, b=0.75):
        self.names = list(docs)
        self.toks = [tokenize(docs[n]) for n in self.names]
        self.avg = sum(map(len, self.toks)) / len(self.toks)
        df = Counter(w for d in self.toks for w in set(d))
        n = len(self.toks)
        self.idf = {w: math.log(1 + (n - f + 0.5) / (f + 0.5)) for w, f in df.items()}
        self.k1, self.b = k1, b

    def search(self, query: str, k=3):
        q = tokenize(query)
        scores = []
        for name, d in zip(self.names, self.toks):
            tf = Counter(d)
            s = sum(self.idf.get(w, 0) * tf[w] * (self.k1 + 1) /
                    (tf[w] + self.k1 * (1 - self.b + self.b * len(d) / self.avg)) for w in q)
            scores.append((s, name))
        return [n for s, n in sorted(scores, reverse=True)[:k] if s > 0]

In [ ]:
question = "How many hours do we have to notify a personal data breach?"
everything = "\n\n".join(f"## {n}\n{t}" for n, t in DOCS.items())
print(f"Stuff the whole library : ~{count_tokens(everything):,} tokens on the desk")
index = BM25(DOCS)
top = index.search(question, k=2)
print(f"Retrieve top-2          : ~{count_tokens(chr(10).join(DOCS[n] for n in top)):,} tokens  -> {top}")

# With Alex: search Alex's own notes instead of reading all of them
notes = {p.name: p.read_text() for p in (lifedb.DATA_DIR / "notes").glob("*.txt")}
print("\nAlex asks about the demo ->", BM25(notes).search("when is demo day for the capstone?", k=1))

## 8. Don't hand-roll everything — what the OpenAI API does for you
*The office does part of it for you* (slides 19–20)

| Feature | In the API | You still |
|---|---|---|
| **Server-side compaction** | `context_management=[{"type": "compaction", "compact_threshold": N}]` | drop items before the latest compaction item |
| **On-demand compaction** | `client.responses.compact(model=..., input=...)` | use the returned window as-is; don't prune it |
| **Prompt caching** | automatic; `usage.input_tokens_details.cached_tokens` | keep instructions and tools identical and **first** |
| **Clearing and memory** | not built in | bring sections 4 and 5 as function tools |

Avoid `truncation="auto"` for real work: it silently drops the oldest items, the papers falling off the desk.
(October 2026, from OpenAI's docs: on GPT-5.6 and later `prompt_cache_key` is optional, and *cache writes* cost
1.25× the input price while cached reads cost 0.1×.)

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 8 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
def drop_before_last_compaction(items):
    """After a compaction item, earlier items are no longer needed (input-array chaining)."""
    idx = [n for n, i in enumerate(items) if i.get("type") == "compaction"]
    return items[idx[-1]:] if idx else items


def compact_now(client, items, model=MODEL):
    """Standalone compaction: you choose the moment (e.g. end of a work phase).
    The returned window is canonical: use it as-is, don't prune it."""
    compacted = client.responses.compact(model=model, input=items)
    return [o.model_dump(exclude_none=True, warnings=False) for o in compacted.output]


PRICE = {"input": 0.10, "cached": 0.01, "output": 0.50}      # gpt-6-luna, $ per 1M tokens


def cost(r) -> float:
    uncached = r.input_tokens - r.cached_tokens
    return (uncached * PRICE["input"] + r.cached_tokens * PRICE["cached"] + r.output_tokens * PRICE["output"]) / 1e6


def stamp(conv, turn):
    conv.system = f"Current time: {time.time():.3f}\n" + SYSTEM     # changes every call: breaks the cache

### 🧪 With Alex: prompt caching
Stable parts first, volatile last. The same review twice: once with a stable start of the desk, once with a
**timestamp at the top of the system prompt**, a classic mistake.

In [ ]:
r_stable = run_agent(REVIEW, life_tools.BINDER_TOOLS, system=SYSTEM, model=MODEL, max_turns=20, verbose=False,
                     extra={"parallel_tool_calls": False, "prompt_cache_key": "alex-review"})
r_stamped = run_agent(REVIEW, life_tools.BINDER_TOOLS, system=SYSTEM, model=MODEL, max_turns=20, verbose=False,
                      extra={"parallel_tool_calls": False}, before_call=stamp)
for label, r in [("stable prefix", r_stable), ("timestamp first", r_stamped)]:
    print(f"{label:16} input {r.input_tokens:6,} · cached {r.cached_tokens:6,} "
          f"({r.cached_tokens / max(r.input_tokens, 1):4.0%}) · ${cost(r):.5f}")
print("\nper step (stable):", [(s['input'], s['cached']) for s in r_stable.steps])

**Discuss:** clearing and compaction also change the desk *before* the newest message. What does that do to the
cache? (Look at run B in section 4: `runs[1]["result"].steps`.) A small desk and a cached desk pull in different directions.

### 🧪 With Alex: compaction by the API (server-side, then on demand)

In [ ]:
class NativeChat(ChatSession):
    def say(self, text):
        answer = super().say(text)
        self.messages = drop_before_last_compaction(self.messages)   # the talk's 08
        return answer

server = NativeChat(native=True, extra={"store": False,
                                        "context_management": [{"type": "compaction", "compact_threshold": 1000}]})
try:
    play_and_probe(server, "4. server-side compaction")
    print("\nitem types on the desk:", [m.get("type") or m.get("role") for m in server.messages])
except Exception as exc:
    print("Server-side compaction is not available for this model/account:", type(exc).__name__, str(exc)[:200])

In [ ]:
long_chat = ChatSession()
for when, text in ALEX_WEEK:
    long_chat.say(f"({when}) {text}")
try:
    window = compact_now(client, long_chat.messages)
    question = [{"role": "user", "content": "(Fri 08:00) What did Priya promise me?"}]
    for label, items in [("full window", long_chat.messages), ("compacted window", window)]:
        r = client.responses.create(model=MODEL, instructions=CHAT_SYSTEM, input=items + question,
                                    max_output_tokens=2000, reasoning={"effort": "low"})
        print(f"{label:17} {len(items):2} items · the API counted {r.usage.input_tokens:,} input tokens → {r.output_text[:90]}")
except Exception as exc:
    print("responses.compact is not available for this model/account:", type(exc).__name__, str(exc)[:200])

**Discuss:** the compaction item is **encrypted**: you can't read or correct the summary. If a detail must survive
for sure, write it to memory (section 5). When would you still hand-roll compaction?

## 9. Everything together — the escalation ladder, live
*The talk's live demo* (slide 18). An agent reads one ~3k-token contract per step, 40 steps, on a 40k desk.
Layers, cheapest first: **offload at the door → clear old results past 50% → compact past 75% → notes in memory survive
all of it.** Runs offline in a few seconds.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# all the code up to section 9 (the same code as the cells above), so you can continue here.
# from checkpoint import *

In [ ]:
class ContextManager:
    def __init__(self, llm, window=40_000, clear_at=0.50, compact_at=0.75, keep_tools=3, mem_dir="./memories_agent"):
        self.window, self.clear_at, self.keep_tools = window, clear_at, keep_tools
        self.compactor = Compactor(llm, window, trigger=compact_at, keep_last=4)
        self.memory = FileMemory(mem_dir)
        self.log = []

    def ingest_tool_output(self, name: str, output: str) -> str:
        return offload_if_large(name, output, max_tokens=1_500)

    def before_model_call(self, messages):
        size = context_size(messages)
        if size > self.clear_at * self.window:
            freed = clear_old_tool_results(messages, keep=self.keep_tools)
            if freed:
                self.log.append(f"clear  : freed ~{freed:,} tokens")
        if self.compactor.needs_compaction(messages):
            before = context_size(messages)
            messages = self.compactor.compact(messages)
            self.log.append(f"compact: {before:,} -> {context_size(messages):,} tokens")
        return messages


def simulate(strategy: str, steps=40, window=40_000):
    """A fake agent that reads one ~3k-token document per step and takes a note."""
    cm = ContextManager(make_llm(real=False), window=window, mem_dir=f"./memories_agent/{strategy}")
    msgs = [{"role": "user", "content": "TASK: review 40 contracts, list any with a penalty clause above 10%."}]
    sizes, overflow_at = [], None
    for i in range(steps):
        doc = f"Contract {i}. Penalty clause {i % 17}%. " + ("Standard legal boilerplate. " * 420)
        msgs.append({"role": "assistant", "content": [
            {"type": "text", "text": f"Reading contract {i}."},
            {"type": "tool_use", "id": f"c{i}", "name": "read_contract", "input": {"id": i}}]})
        body = cm.ingest_tool_output("read_contract", doc) if strategy == "offload" else doc
        msgs.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": f"c{i}", "content": body}]})
        if i % 17 > 10:  # the agent writes findings to memory, outside the window
            cm.memory.handle({"command": "append", "path": "/memories/findings.md",
                              "text": f"- contract {i}: penalty {i % 17}%"})
        if strategy in ("layered", "offload"):
            msgs = cm.before_model_call(msgs)
        size = context_size(msgs)
        sizes.append(size)
        if size > window and overflow_at is None:
            overflow_at = i
    return sizes, overflow_at, cm


def bar(v, window, width=46):
    n = min(width, int(width * v / window))
    return "#" * n + ("!" if v > window else "")

In [ ]:
for d in ("./memories_agent", "./.tool_outputs"):
    shutil.rmtree(d, ignore_errors=True)
W = 40_000
results = {s: simulate(s, window=W) for s in ("none", "layered", "offload")}

print(f"Context size per step (window = {W:,} tokens; '!' = desk overflowed)\n")
print(f"{'step':>4}  {'no management':<48}{'clear (+compact)':<48}offload at the door")
for step in range(0, 40, 3):
    print(f"{step:>4}  " + "".join(f"{bar(results[s][0][step], W):<48}" for s in ("none", "layered", "offload")))
for s, (sizes, overflow, cm) in results.items():
    print(f"\n{s:<8} peak {max(sizes):>7,} tokens | overflow at step {overflow if overflow is not None else '-'}")
    if cm.log:
        n_clear = sum(l.startswith("clear") for l in cm.log)
        n_comp = sum(l.startswith("compact") for l in cm.log)
        print(f"          {n_clear} clearing events, {n_comp} compactions; e.g. {cm.log[0]}")
print("\nMemory survived everything:")
print(results["layered"][2].memory.handle({"command": "view", "path": "/memories/findings.md"}))

## Takeaways

**Beyond the basics** (slide 20): caching and order (stable first) · tool design (fewer tools, concise outputs) ·
security (tool results and memories are untrusted input) · privacy (memory is personal data) · evals (log the desk
per turn, probe what survives compaction) · cost and latency (a cheap model for summaries).

**The one-minute version** (slide 21): *It only sees what's on the desk. Every turn it re-reads it all: more paper,
more cost. Pin the brief in the corner. Leave a sticky note, not the binder. Before papers fall off, summarise what
matters. Keep tomorrow's notes in a drawer. Bring three pages, not the library. Send an intern to the archive.*

**Six habits** (slide 22): measure first · pin and mark · cheapest first (offload → clear → compact → reset) ·
write it down · isolate the mess · govern it.

## Homework

1. **Package it.** Move the notebook code into a `context_toolkit/` folder laid out like the talk's companion code:
   `common.py`, `01_token_budget.py` … `09_context_manager.py`, each runnable on its own (`python 03_compaction.py`).
   Add `tests/` that run **without an API key** (use `fake_llm`). At least: the four ladder thresholds; the window keeps
   the pinned brief and never starts with an orphan tool result; compaction keeps the last N messages and passes the
   previous summary in; clearing keeps the newest N results and never touches `memory`; the memory path guard blocks
   both `../../etc/passwd` and `../memories_evil`.
2. **Alex's ladder.** Use `ContextManager`'s ideas as a `before_call` for Alex's 8-week review on the real API:
   clear past 50%, compact past 75% (OpenAI items: `clear_old_function_outputs`, and a summary of older items).
   Report the peak desk and whether the answer is still right.

**Stretch:** pinned facts that compaction may never drop · measure the cache cost of clearing (run A vs B) ·
`load_relevant_memories` for the chatbot in section 3.